# Fase 1 — Preparação e Serialização do Modelo

## Objetivo

Nesta etapa, será reaproveitado o modelo **Support Vector Machine (SVM)** desenvolvido no mini-projeto anterior de classificação de dígitos manuscritos utilizando a base de dados MNIST.

O SVM apresentou acurácia de aproximadamente **98,3% no conjunto de teste**, superando os outros modelos avaliados naquele projeto. Por esse motivo, ele foi escolhido para ser disponibilizado por meio de uma API desenvolvida com FastAPI.

## O que será realizado?

1. Carregar a base de dados MNIST.
2. Separar os dados em conjuntos de treinamento, validação e teste.
3. Normalizar os pixels das imagens para a faixa de 0 a 1.
4. Treinar o modelo SVM com os parâmetros definidos no mini-projeto.
5. Avaliar seu desempenho com métricas de classificação.
6. Serializar o modelo em um arquivo `.joblib`, permitindo que a API o carregue posteriormente sem precisar treiná-lo novamente.

## Por que serializar o modelo?

O treinamento de um algoritmo de Machine Learning pode exigir tempo e recursos computacionais. A serialização permite salvar o modelo treinado em um arquivo e reutilizá-lo em outras aplicações.

Assim, a API poderá carregar o modelo durante sua inicialização e utilizá-lo para classificar novos dígitos, sem executar novamente o processo de treinamento a cada requisição.

**Observação:** o modelo será treinado com imagens cujos pixels estejam normalizados entre 0 e 1. Por isso, os dados recebidos pela API deverão seguir o mesmo padrão de entrada.

In [1]:

from pathlib import Path
import json
import time
import joblib
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Criar a pasta onde o modelo será salvo
PASTA_MODELOS = Path("models")
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

# 2. Carregar a base de dados MNIST
dados = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False,
    parser="auto"
)

X = dados.data
y = dados.target.astype(int)

# Garantir que os rótulos sejam numéricos
y = y.astype(int)

print("Dimensões das imagens:", X.shape)
print("Quantidade de rótulos:", len(y))

# 3. Separar os dados em treino, validação e teste
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.125,
    random_state=42,
    stratify=y_train_val
)

# 4. Normalizar os pixels para a faixa de 0 a 1
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

print("Treinamento:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)

# 5. Criar e treinar o modelo SVM
modelo_svm = SVC(
    C=10,
    kernel="rbf"
)

inicio = time.time()
modelo_svm.fit(X_train, y_train)
tempo_treinamento = time.time() - inicio

# 6. Avaliar o modelo no conjunto de teste
y_pred = modelo_svm.predict(X_test)

metricas = {
    "modelo": "SVM",
    "acuracia": float(accuracy_score(y_test, y_pred)),
    "precisao_weighted": float(
        precision_score(y_test, y_pred, average="weighted")
    ),
    "recall_weighted": float(
        recall_score(y_test, y_pred, average="weighted")
    ),
    "f1_weighted": float(
        f1_score(y_test, y_pred, average="weighted")
    ),
    "tempo_treinamento_segundos": round(tempo_treinamento, 2)
}

print("\nMétricas do modelo:")
for nome, valor in metricas.items():
    print(f"{nome}: {valor}")

# 7. Salvar o modelo treinado
caminho_modelo = PASTA_MODELOS / "svm_mnist.joblib"
joblib.dump(modelo_svm, caminho_modelo)

# 8. Salvar as métricas em um arquivo JSON
caminho_metricas = PASTA_MODELOS / "metricas.json"

with open(caminho_metricas, "w", encoding="utf-8") as arquivo:
    json.dump(metricas, arquivo, indent=4, ensure_ascii=False)

print("\nModelo salvo em:", caminho_modelo)
print("Métricas salvas em:", caminho_metricas)


Dimensões das imagens: (70000, 784)
Quantidade de rótulos: 70000
Treinamento: (49000, 784)
Validação: (7000, 784)
Teste: (14000, 784)

Métricas do modelo:
modelo: SVM
acuracia: 0.983
precisao_weighted: 0.9830040759001173
recall_weighted: 0.983
f1_weighted: 0.9829912909280121
tempo_treinamento_segundos: 362.76

Modelo salvo em: models\svm_mnist.joblib
Métricas salvas em: models\metricas.json


## Resultados da Fase 1

O modelo SVM foi treinado com 49.000 imagens do conjunto MNIST, utilizando os parâmetros `C=10` e `kernel="rbf"`. Os dados foram divididos em treinamento (70%), validação (10%) e teste (20%), preservando a proporção das classes por meio do parâmetro `stratify`.

Após a normalização dos pixels, o modelo foi avaliado em 14.000 imagens reservadas para teste. Foram obtidos os seguintes resultados:

- **Acurácia:** 98,30%.
- **Precisão ponderada:** aproximadamente 98,30%.
- **Recall ponderado:** 98,30%.
- **F1-score ponderado:** aproximadamente 98,30%.
- **Tempo de treinamento:** 206,84 segundos.

Os resultados indicam bom desempenho na classificação de dígitos manuscritos. O tempo de treinamento também foi registrado para documentar o custo computacional da preparação do modelo.

Por fim, o modelo foi serializado no arquivo `models/svm_mnist.joblib`, enquanto as métricas foram armazenadas em `models/metricas.json`. Esses arquivos permitem reaproveitar o modelo sem precisar treiná-lo novamente durante cada inicialização da API, embora o artefato ainda precise ser carregado na memória quando o serviço iniciar.

In [2]:

#confirmar que o modelo salvo pode ser carregado e utilizado para prever uma imagem.

modelo_carregado = joblib.load("models/svm_mnist.joblib")

# Utiliza uma imagem do conjunto de teste
imagem_teste = X_test[0].reshape(1, -1)
previsao = modelo_carregado.predict(imagem_teste)[0]

print("Modelo carregado com sucesso!")
print("Dígito previsto:", previsao)
print("Dígito correto:", y_test[0])

Modelo carregado com sucesso!
Dígito previsto: 7
Dígito correto: 7


# Fase 2 — Estrutura modular da API FastAPI
A API será organizada em módulos com responsabilidades específicas, seguindo o princípio de separação de responsabilidades.

- **main.py:** ponto de entrada da aplicação, responsável por configurar o FastAPI, registrar as rotas e gerenciar o ciclo de vida do serviço.
- **schemas.py:** define os modelos Pydantic que validam os dados recebidos e as respostas produzidas pela API.
- **services.py:** concentra a lógica de inferência, utilizando o modelo de Machine Learning serializado.
- **routers.py:** define os endpoints HTTP, incluindo o monitoramento de saúde e a realização de previsões.

Essa estrutura facilita a manutenção, os testes e a evolução da aplicação. Também evita misturar a lógica do modelo com o tratamento das requisições HTTP.

O modelo SVM desenvolvido anteriormente será reaproveitado como artefato preditivo. Nas próximas etapas, será integrado à API para permitir a classificação de imagens por meio de requisições HTTP.

In [3]:
# Criar a estrutura de pastas

from pathlib import Path

pastas = [
    Path("app"),
    Path("models"),
    Path("notebooks")
]

for pasta in pastas:
    pasta.mkdir(parents=True, exist_ok=True)

# Cria o arquivo que identifica app como pacote Python
(Path("app") / "__init__.py").touch()

print("Estrutura inicial criada.")

Estrutura inicial criada.


# Instalar as dependências da API
    # python -m pip install fastapi uvicorn

# conferir se as bibliotecas estão disponíveis:
    # python -c "import fastapi, uvicorn; print('FastAPI e Uvicorn OK')"

FastAPI: biblioteca usada para criar a API REST.  
Uvicorn: servidor que executa a aplicação e recebe as requisições HTTP.

In [4]:
# Fase 2.1 — Criar o arquivo schemas.py
# O arquivo schemas.py será responsável por definir quais dados a API aceita e qual formato ela devolve.

# No nosso projeto, o modelo reconhece dígitos manuscritos de 0 a 9. Cada imagem precisa ser representada por 784 valores de pixels, 
# correspondentes a uma imagem de 28 × 28 pixels.

New-Item .\app\schemas.py -ItemType File

SyntaxError: unexpected character after line continuation character (2415557730.py, line 7)